# Chapter 23 — Blind Before You Compare

**Companion to *Applied AI*.**

Three reviewers, one paragraph. If the second reviewer read the first
reviewer's answer before writing, their agreement means something different
from agreement without exposure. Before variety can be measured, proposals
must be collected blind — none able to see another.

## Question

**What does a seal exclude, and what walks straight through it?**

## What this notebook does

It **inspects** the two preserved boundary bundles — the sealed fan-out run
(`sealed-proposals/2026-09-14-a1b562a/`, six cases) and its byte-level
extension (`sealed-rendered/2026-09-14-1b3c7a2/`, four cases): the clean
fan-out, the loud refusals, and the two admitted boundaries, proved down to
the sent bytes.

```text
blind  ≠  independent  ≠  diverse
```

## Setup

Standard library only. No network, no API key, no `codeai` import.
Only bundle-relative paths are shown; override the evidence root with
`APPLIED_AI_EVIDENCE`.

In [1]:
import json
import os
from pathlib import Path

def find_evidence_dir(marker="sealed-proposals"):
    """Locate the preserved evidence. Override with APPLIED_AI_EVIDENCE."""
    env = os.environ.get("APPLIED_AI_EVIDENCE")
    if env and Path(env).expanduser().is_dir():
        return Path(env).expanduser()
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base / "evidence",
                     base / "experiments" / "applied-ai" / "evidence"):
            if (cand / marker).is_dir():
                return cand
    raise FileNotFoundError(
        "Preserved evidence not found. Set APPLIED_AI_EVIDENCE to the "
        "directory holding the Applied AI evidence bundles.")

EVIDENCE_DIR = find_evidence_dir()
SP = EVIDENCE_DIR / "sealed-proposals" / "2026-09-14-a1b562a"
SR = EVIDENCE_DIR / "sealed-rendered" / "2026-09-14-1b3c7a2"
print("bundles: sealed-proposals/2026-09-14-a1b562a")
print("         sealed-rendered/2026-09-14-1b3c7a2")
fan = json.loads((SP / "results.json").read_text(encoding="utf-8"))
byt = json.loads((SR / "results.json").read_text(encoding="utf-8"))
sra = json.loads((SR / "analysis.json").read_text(encoding="utf-8"))
bodies = json.loads((SR / "transport_bodies.json").read_text(encoding="utf-8"))
print("fan-out cases:", sorted(fan), "| byte-level cases:", sorted(byt))

bundles: sealed-proposals/2026-09-14-a1b562a
         sealed-rendered/2026-09-14-1b3c7a2
fan-out cases: ['branch-failure', 'clean-fanout', 'copied-text', 'declared-artifact', 'forbidden-lineage', 'omitted-provenance'] | byte-level cases: ['copied-prompt', 'omitted-included', 'sealed-excluded', 'tamper-refused']


## 1. The clean fan-out

Three branches, one shared base prompt — each branch's seal forbids exactly
its siblings. Both recorded packages carry no sibling sentinel. Blinding is
the arrangement; it says nothing yet about whether the branches will agree or
differ (that is Chapter 24's measurement, not this chapter's).

In [2]:
cf = fan["clean-fanout"]
print("clean fan-out branch statuses:", cf["statuses"])
assert cf["statuses"] == ["succeeded", "succeeded"]
print()
print("Two branches ran blind. What they produced — and whether the extra")
print("proposal bought anything — is a separate, measured question.")

clean fan-out branch statuses: ['succeeded', 'succeeded']

Two branches ran blind. What they produced — and whether the extra
proposal bought anything — is a separate, measured question.


## 2. What the seal excludes — loudly

A declared artifact under seal, or a forbidden sibling lineage, does not get
silently dropped on the fan-out path. Every explicitly passed base input is a
*required* candidate, so a seal conflict raises `RequiredContextMissing`
before any branch executes: zero branch executions, no package, no call, no
manifest.

In [3]:
for case in ("declared-artifact", "forbidden-lineage"):
    r = fan[case]
    print(f"{case}:")
    print(f"  raised: {r['raised']} | branch executions: {r['branch_executions']}")
    print(f"  error : {r['error'][:100]}...")
    assert r["raised"] and r["branch_executions"] == [0, 0]
print()
print("The refusal is loud by design: a protocol amendment, recorded before")
print("any result existed, changed 'silently exclude' into 'fail before any")
print("branch runs'. Silent exclusion lives one layer down, at the compiler.")

declared-artifact:
  raised: True | branch executions: [0, 0]
  error : RequiredContextMissing: REQUIRED_CONTEXT_MISSING: d539275ddfe0813313c69d231823df5d76a93f5ecc2608b376...
forbidden-lineage:
  raised: True | branch executions: [0, 0]
  error : RequiredContextMissing: REQUIRED_CONTEXT_MISSING: 523f0500-b6d2-4066-a044-02934368a3af required but ...

The refusal is loud by design: a protocol amendment, recorded before
any result existed, changed 'silently exclude' into 'fail before any
branch runs'. Silent exclusion lives one layer down, at the compiler.


## 3. What walks straight through, part one: omitted provenance

The same artifact ID with its lineage map omitted is **included** — by
reference, bytes retrievable from the store. The seal had nothing to match
against. Down to the bytes: the sentinel is present in the rendered bytes
*and* in the sent body.

In [4]:
op = fan["omitted-provenance"]
print("fan-out omitted-provenance:", op["statuses"], "(ran — boundary, kept)")
assert op["statuses"] == ["succeeded", "succeeded"]

om = sra["omitted-included"]
print("rendered bytes contain sentinel:", om["sentinel_in_bytes"])
assert om["sentinel_in_bytes"] is True
sent_omitted = json.dumps(bodies["call-omitted"])
print("sent body contains sentinel   :", "S3NT1N3L" in sent_omitted)
assert "S3NT1N3L" in sent_omitted
print()
print("PRESERVED NEGATIVE RESULT: seals block declared channels, not every")
print("information-flow channel. Provenance is part of the enforcement")
print("mechanism — when the caller never declares it, the seal loses what it")
print("needs. An observation without provenance is not attributable evidence.")

fan-out omitted-provenance: ['succeeded', 'succeeded'] (ran — boundary, kept)
rendered bytes contain sentinel: True
sent body contains sentinel   : True

PRESERVED NEGATIVE RESULT: seals block declared channels, not every
information-flow channel. Provenance is part of the enforcement
mechanism — when the caller never declares it, the seal loses what it
needs. An observation without provenance is not attributable evidence.


## 4. What walks straight through, part two: copied text

Sibling text copied into the shared prompt is not filtered — seals match
declared identifiers, never arbitrary semantics. Byte-level proof is two-sided:
the render (context items only) is clean, while the sent body carries the
sentinel in via the branch prompt.

In [5]:
cp = fan["copied-text"]
print("fan-out copied-text:", cp["statuses"], "(ran — boundary, kept)")
assert cp["statuses"] == ["succeeded", "succeeded"]

cr = sra["copied-prompt"]
print("rendered bytes contain sentinel:", cr["sentinel_in_bytes"])
sent_copied = json.dumps(bodies["call-copied"])
print("sent body contains sentinel   :", "S3NT1N3L" in sent_copied)
assert cr["sentinel_in_bytes"] is False
assert "S3NT1N3L" in sent_copied
print()
print("The seal is an identifier filter with durable reasons — not a semantic")
print("firewall, not access control, not sandboxing. Excluded-from-selection")
print("content could still be read directly from storage.")

fan-out copied-text: ['succeeded', 'succeeded'] (ran — boundary, kept)
rendered bytes contain sentinel: False
sent body contains sentinel   : True

The seal is an identifier filter with durable reasons — not a semantic
firewall, not access control, not sandboxing. Excluded-from-selection
content could still be read directly from storage.


## 5. Tampering is refused; failure stays local

A prepared body that does not carry the rendered input is refused before any
provider effect (`RenderBindingError`, zero sends). And when one branch fails,
the fan-out does not: F1 succeeded with output intact, F2 failed with the
error preserved, `fanout.completed` holds both.

In [6]:
tr = byt["tamper-refused"]
print("tamper: raised =", tr["raised"], "| sends =", tr["sends"])
assert tr["raised"] and tr["sends"] == 0

bf = fan["branch-failure"]
print("branch-failure:", bf["statuses"], "| F2 adapter calls:", bf["F2_calls"])
assert bf["statuses"] == ["succeeded", "failed"] and bf["F2_calls"] == 1
assert sra["tamper-refused"].startswith("RenderBindingError")
print()
print("Binding holds at the send boundary; isolation holds across branches.")
print("Neither makes the branches diverse — blinding protects experimental")
print("independence from information leakage. It does not guarantee")
print("uncorrelated failures. That measurement is Chapter 24.")

tamper: raised = True | sends = 0
branch-failure: ['succeeded', 'failed'] | F2 adapter calls: 1

Binding holds at the send boundary; isolation holds across branches.
Neither makes the branches diverse — blinding protects experimental
independence from information leakage. It does not guarantee
uncorrelated failures. That measurement is Chapter 24.


## Interpretation

1. **Blind ≠ independent ≠ diverse.** The seal implements the first: no
   branch sees another's declared lineage. Independence of failures and
   diversity of coverage are measured afterwards, never assumed.
2. **The seal excludes declared siblings, loudly.** Required candidates in
   conflict fail before any branch executes.
3. **Omitted provenance is admitted.** The same artifact without its lineage
   map renders into the bytes the provider would receive.
4. **Copied text walks through.** Identifier matching never sees semantics;
   the sent body proves it.
5. **Sealed is not sandboxed.** Selection does not dereference sources and
   does not enforce downstream access.

## Try it yourself

1. In `transport_bodies.json`, find the `<context-item>` for the omitted
   artifact in `call-omitted`: is the sentinel inlined or referenced? What
   does "included by reference" mean for a reader of the ledger?
2. The fan-out protocol was amended mid-run (silent exclusion → loud
   refusal). Find the amendment note in the bundle's `preregistration.json`
   — why was the loud behavior the honest one?
3. Suppose two blind branches agree. Name two histories that produce that
   agreement, and the measurement that tells them apart. (Chapter 24 runs it.)

*Evidence: `experiments/applied-ai/evidence/sealed-proposals/2026-09-14-a1b562a/`
and `experiments/applied-ai/evidence/sealed-rendered/2026-09-14-1b3c7a2/`
(pinned runs, stdlib-only verifiers, seeded corruptions rejected). No
network, no API key, no `codeai` import.*